# Figure 4: Simulated batch integration

Run after the training and evaluation commands in `bash/paper/`. SCENE outputs use the `scLDM` names referenced below.


In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment_scene.yaml").exists())
os.chdir(PROJECT_ROOT / "notebooks" / "figures")
for folder in ("fig_1", "fig_2", "fig_3", "fig_4", "fig_5", "fig_6", "app", "qc"):
    Path("output", folder).mkdir(parents=True, exist_ok=True)


## Figure 4 | Batch Integration

In [ ]:
import anndata as ad
import numpy as np

adata = ad.read_h5ad("../../data/sim2_norm.h5ad")

In [ ]:
import scanpy as sc

_, n_cells = sc.pp.filter_genes(adata, min_cells=1, inplace=False)
adata.var["n_cells"] = n_cells

In [ ]:
# Load scLDM latent representations
z_cells_scldm = np.load('../../results/sim_2/scLDM/scLDM_cell_latent.npy')
z_genes_scldm = np.load('../../results/sim_2/scLDM/scLDM_gene_latent.npy')

if z_cells_scldm.shape[1] != z_genes_scldm.shape[1]:
    raise ValueError(f"Latent dimension mismatch: cells={z_cells_scldm.shape}, genes={z_genes_scldm.shape}")

adata.obsm['scLDM'] = z_cells_scldm
adata.varm['scLDM'] = z_genes_scldm


# Load scLDM batch latent representations
z_cells_scldm_batch = np.load('../../results/sim_2/scLDM_batch_full/scLDM_cell_latent.npy')
z_genes_scldm_batch = np.load('../../results/sim_2/scLDM_batch_full/scLDM_gene_latent.npy')

if z_cells_scldm_batch.shape[1] != z_genes_scldm_batch.shape[1]:
    raise ValueError(f"Latent dimension mismatch: cells={z_cells_scldm_batch.shape}, genes={z_genes_scldm_batch.shape}")

adata.obsm['scLDM_batch'] = z_cells_scldm_batch
adata.varm['scLDM_batch'] = z_genes_scldm_batch


# Load scLDM batch subbatch latent representations
z_cells_scldm_batch_subbatch = np.load('../../results/sim_2/scLDM_batch_full_lowrank_4/scLDM_cell_latent.npy')
z_genes_scldm_batch_subbatch = np.load('../../results/sim_2/scLDM_batch_full_lowrank_4/scLDM_gene_latent.npy')

if z_cells_scldm_batch_subbatch.shape[1] != z_genes_scldm_batch_subbatch.shape[1]:
    raise ValueError(f"Latent dimension mismatch: cells={z_cells_scldm_batch_subbatch.shape}, genes={z_genes_scldm_batch_subbatch.shape}")

adata.obsm['scLDM_batch_subbatch'] = z_cells_scldm_batch_subbatch
adata.varm['scLDM_batch_subbatch'] = z_genes_scldm_batch_subbatch


In [ ]:
# Load scVI latent representations
z_cells_scvi = np.load('../../results/sim_2/scVI/scVI_cell_latent.npy')

adata.obsm['scVI'] = z_cells_scvi


z_cells_scvi_batch = np.load('../../results/sim_2/scVI_batch/scVI_cell_latent.npy')

adata.obsm['scVI_batch'] = z_cells_scvi_batch


z_cells_scvi_batch_subbatch = np.load('../../results/sim_2/scVI_batch_subbatch/scVI_cell_latent.npy')

adata.obsm['scVI_batch_subbatch'] = z_cells_scvi_batch_subbatch


In [ ]:
import scanpy as sc
sc.pp.neighbors(adata, use_rep="scLDM")
sc.tl.umap(adata, random_state=42, key_added="umap_scLDM")

In [ ]:
import scanpy as sc
sc.pp.neighbors(adata, use_rep="scLDM_batch")
sc.tl.umap(adata, random_state=42, key_added="umap_scLDM_batch")

In [ ]:
import scanpy as sc
sc.pp.neighbors(adata, use_rep="scLDM_batch_subbatch")
sc.tl.umap(adata, random_state=42, key_added="umap_scLDM_batch_subbatch")

In [ ]:
import scanpy as sc
sc.pp.neighbors(adata, use_rep="scVI")
sc.tl.umap(adata, random_state=42, key_added="umap_scVI")

In [ ]:
import scanpy as sc
sc.pp.neighbors(adata, use_rep="scVI_batch")
sc.tl.umap(adata, random_state=42, key_added="umap_scVI_batch")

In [ ]:
import scanpy as sc
sc.pp.neighbors(adata, use_rep="scVI_batch_subbatch")
sc.tl.umap(adata, random_state=42, key_added="umap_scVI_batch_subbatch")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal

keys_to_plot = ["Group", "Batch", "SubBatch"]
palettes = {}

for key in keys_to_plot:
    group = adata.obs[key]
    categories = pd.Categorical(group).categories
    n_cat = len(categories)

    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]

    palettes[key] = dict(zip(categories, colors))

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal
import pandas as pd

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


# Extract coordinates
umap = adata.obsm["umap_scLDM"]
keys_to_plot = ["Group", "Batch", 'SubBatch']

for key in keys_to_plot:
    group = adata.obs[key]
    categories = pd.Categorical(group).categories
    palette = palettes[key]

    fig, ax = plt.subplots(figsize=(1.2, 1.))


    # Plot umap by cell type
    for ct in categories:
        idx = (group == ct).values
        ax.scatter(
            umap[idx, 0],
            umap[idx, 1],
            s=0.3,
            c=[palette[ct]],
            alpha=1,
            #marker='.',
            label=ct,
            linewidths=0,
            rasterized=True
        )

    ax.set_axis_off()
    ax.set_title("")

    plt.savefig(f"output/fig_4/sim2_umap_scldm_{key}.svg", bbox_inches="tight", pad_inches=0, dpi=600)
    plt.show()
    #plt.close()

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal
import pandas as pd

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

# Extract coordinates
umap = adata.obsm["umap_scLDM_batch"]
keys_to_plot = ["Group", "Batch", 'SubBatch']

for key in keys_to_plot:
    group = adata.obs[key]
    categories = pd.Categorical(group).categories
    palette = palettes[key]

    fig, ax = plt.subplots(figsize=(1.2, 1.))


    # Plot umap by cell type
    for ct in categories:
        idx = (group == ct).values
        ax.scatter(
            umap[idx, 0],
            umap[idx, 1],
            s=0.3,
            c=[palette[ct]],
            alpha=1,
            #marker='.',
            label=ct,
            linewidths=0,
            rasterized=True
        )

    ax.set_axis_off()
    ax.set_title("")

    plt.savefig(f"output/fig_4/sim2_umap_scldm_batch_{key}.svg", bbox_inches="tight", pad_inches=0, dpi=600)
    plt.show()
    #plt.close()

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal
import pandas as pd

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

# Extract coordinates
umap = adata.obsm["umap_scLDM_batch_subbatch"]
keys_to_plot = ["Group", "Batch", 'SubBatch']

for key in keys_to_plot:
    group = adata.obs[key]
    categories = pd.Categorical(group).categories
    palette = palettes[key]

    fig, ax = plt.subplots(figsize=(1.2, 1.))


    # Plot umap by cell type
    for ct in categories:
        idx = (group == ct).values
        ax.scatter(
            umap[idx, 0],
            umap[idx, 1],
            s=0.3,
            c=[palette[ct]],
            alpha=1,
            #marker='.',
            label=ct,
            linewidths=0,
            rasterized=True
        )

    ax.set_axis_off()
    ax.set_title("")

    plt.savefig(f"output/fig_4/sim2_umap_scldm_batch_subbatch_{key}.svg", bbox_inches="tight", pad_inches=0, dpi=600)
    plt.show()
    #plt.close()

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal
import pandas as pd

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

# Extract coordinates
umap = adata.obsm["umap_scVI"]
keys_to_plot = ["Group", "Batch", 'SubBatch']

for key in keys_to_plot:
    group = adata.obs[key]
    categories = pd.Categorical(group).categories
    palette = palettes[key]

    fig, ax = plt.subplots(figsize=(1.2, 1.))


    # Plot umap by cell type
    for ct in categories:
        idx = (group == ct).values
        ax.scatter(
            umap[idx, 0],
            umap[idx, 1],
            s=0.3,
            c=[palette[ct]],
            alpha=1,
            #marker='.',
            label=ct,
            linewidths=0,
            rasterized=True
        )
    ax.set_axis_off()
    ax.set_title("")

    plt.savefig(f"output/fig_4/sim2_umap_scvi_{key}.svg", bbox_inches="tight", pad_inches=0, dpi=600)
    plt.show()
    #plt.close()

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal
import pandas as pd

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

# Extract coordinates
umap = adata.obsm["umap_scVI_batch"]
keys_to_plot = ["Group", "Batch", 'SubBatch']

for key in keys_to_plot:
    group = adata.obs[key]
    categories = pd.Categorical(group).categories
    palette = palettes[key]

    fig, ax = plt.subplots(figsize=(1.2, 1.))

    # Plot umap by cell type
    for ct in categories:
        idx = (group == ct).values
        ax.scatter(
            umap[idx, 0],
            umap[idx, 1],
            s=0.3,
            c=[palette[ct]],
            alpha=1,
            #marker='.',
            label=ct,
            linewidths=0,
            rasterized=True
        )

    ax.set_axis_off()
    ax.set_title("")

    plt.savefig(f"output/fig_4/sim2_umap_scvi_batch_{key}.svg", bbox_inches="tight", pad_inches=0, dpi=600)
    plt.show()
    #plt.close()

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal
import pandas as pd

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

# Extract coordinates
umap = adata.obsm["umap_scVI_batch_subbatch"]
keys_to_plot = ["Group", "Batch", 'SubBatch']

for key in keys_to_plot:
    group = adata.obs[key]
    categories = pd.Categorical(group).categories
    palette = palettes[key]

    fig, ax = plt.subplots(figsize=(1.2, 1.))


    # Plot umap by cell type
    for ct in categories:
        idx = (group == ct).values
        ax.scatter(
            umap[idx, 0],
            umap[idx, 1],
            s=0.3,
            c=[palette[ct]],
            alpha=1,
            #marker='.',
            label=ct,
            linewidths=0,
            rasterized=True
        )

    ax.set_axis_off()
    ax.set_title("")

    plt.savefig(f"output/fig_4/sim2_umap_scvi_batch_subbatch_{key}.svg", bbox_inches="tight", pad_inches=0, dpi=600)
    plt.show()
    #plt.close()

In [ ]:
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def save_cluster_legend_pdf(
    categories,
    palette,
    output_pdf,
    ncol=1,
    fontsize=6,
    marker_size=4.0,
    columnspacing=0.8,
    handletextpad=0.4,
    labelspacing=0.35,
    borderpad=0.2,
):
    """
    Save a standalone legend-only PDF for cluster colors.

    Parameters
    ----------
    categories : list[str]
        Ordered category names.
    palette : dict
        Mapping {category: color}.
    output_pdf : str
        Output path ending in .pdf.
    ncol : int
        Number of legend columns.
    fontsize : float
        Legend text size in pt. Nature-style target: ~5–7 pt.
    marker_size : float
        Marker size in pt for legend keys.
    """

    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    })


    handles = [
        Line2D(
            [0], [0],
            linestyle="None",
            marker="o",
            markersize=marker_size,
            markerfacecolor=palette[cat],
            markeredgecolor=palette[cat],
            markeredgewidth=0.0,
            label=str(cat),
        )
        for cat in categories
    ]

    # Rough figure size estimate so the legend lays out predictably.
    n_items = len(categories)
    n_rows = math.ceil(n_items / ncol)
    fig_w = max(1.2, 1.15 * ncol + 0.55 * ncol)
    fig_h = max(0.35, 0.22 * n_rows + 0.18)

    fig = plt.figure(figsize=(fig_w, fig_h))
    fig.legend(
        handles=handles,
        labels=categories,
        loc="center",
        ncol=ncol,
        frameon=False,
        fontsize=fontsize,
        handlelength=0.8,
        handletextpad=handletextpad,
        columnspacing=columnspacing,
        labelspacing=labelspacing,
        borderpad=borderpad,
        markerscale=1.0,
    )

    fig.savefig(
        output_pdf,
        format="svg",
        bbox_inches="tight",
        pad_inches=0.01,
        transparent=True,
    )
    plt.close(fig)

In [ ]:
categories = list(adata.obs["Group"].cat.categories)

save_cluster_legend_pdf(
    categories=categories,
    palette=palettes["Group"],
    output_pdf="output/fig_4/sim2_umap_cluster_legend.svg",
    ncol=4,          # Number of legend columns
    fontsize=6,
    marker_size=4.0,
)